In [1]:
import numpy as np
from scipy import stats
from pathlib import Path

Setup and data

In [2]:
np.random.seed(0)

c = 3e8          # Speed of light (m/s)
fc = 3.5e9       # Carrier frequency (Hz)
d0 = 1.0         # Reference distance (m)

FSPL_d0_theory = 20 * np.log10(4 * np.pi * d0 * fc / c)


def synthesize_pathloss(
    n_true,
    sigma_true,
    n_samples=150,
    d_min=10.0,
    d_max=1800.0,
    seed=1
):
    """Generate synthetic path-loss measurements using the CI model."""
    rng = np.random.default_rng(seed)

    log_d = rng.uniform(
        np.log10(d_min),
        np.log10(d_max),
        size=n_samples
    )
    d = 10 ** log_d

    shadowing = rng.normal(0, sigma_true, size=n_samples)

    PL = (
        FSPL_d0_theory
        + 10 * n_true * np.log10(d / d0)
        + shadowing
    )

    return d, PL


DATA_PATH = Path("pathloss.txt")

# Used only if pathloss.txt is not available.
N_TRUE = 3.0
SIGMA_TRUE = 6.0


if DATA_PATH.exists():
    data = np.loadtxt(DATA_PATH)

    # Validate the input file before using it.
    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError(
            "pathloss.txt must contain at least two columns: "
            "distance (m) and path loss (dB)."
        )

    d_meas = data[:, 0]
    PL_meas = data[:, 1]

    print(
        f"Loaded REAL measurement file: "
        f"{DATA_PATH} ({len(d_meas)} points)."
    )

else:
    d_meas, PL_meas = synthesize_pathloss(
        N_TRUE,
        SIGMA_TRUE
    )

    print(
        f"[Fallback] '{DATA_PATH}' not found — using a synthetic "
        f"measurement campaign "
        f"(true n = {N_TRUE}, true sigma = {SIGMA_TRUE} dB, "
        f"N = {len(d_meas)} points, "
        f"d in [{d_meas.min():.1f}, {d_meas.max():.1f}] m)."
    )

print(
    f"Theoretical FSPL(d0) = "
    f"{FSPL_d0_theory:.3f} dB"
)


[Fallback] 'pathloss.txt' not found — using a synthetic measurement campaign (true n = 3.0, true sigma = 6.0 dB, N = 150 points, d in [10.3, 1764.3] m).
Theoretical FSPL(d0) = 43.323 dB


Linear Regression

In [3]:
class MyLinearRegression:
    """
    Simple linear regression:
        y = theta_0 + theta_1 * x

    Implemented from scratch using NumPy.
    """

    def __init__(self):
        self.theta = None
        self.loss_history_ = None

    @staticmethod
    def _design_matrix(x):
        x = np.asarray(x).reshape(-1)
        return np.column_stack([
            np.ones_like(x),
            x
        ])

    @staticmethod
    def _mse(Phi, y, theta):
        residual = Phi @ theta - y
        return np.mean(residual ** 2)

    def fit(
        self,
        x,
        y,
        method="closed_form",
        lr=0.05,
        epochs=1500,
        random_state=0
    ):
        Phi = self._design_matrix(x)
        y = np.asarray(y).reshape(-1)

        if len(Phi) != len(y):
            raise ValueError("x and y must contain the same number of samples.")

        n = len(y)

        if method == "closed_form":
            # Normal equation:
            # theta = (Phi^T Phi)^(-1) Phi^T y
            self.theta = np.linalg.solve(
                Phi.T @ Phi,
                Phi.T @ y
            )
            self.loss_history_ = None

        elif method == "batch_gd":
            theta = np.zeros(2)
            history = np.empty(epochs)

            for epoch in range(epochs):
                residual = Phi @ theta - y
                grad = (2.0 / n) * (Phi.T @ residual)

                theta -= lr * grad

                # Stop safely if numerical instability occurs.
                if not np.all(np.isfinite(theta)):
                    raise FloatingPointError(
                        "Batch gradient descent became numerically unstable. "
                        "Try a smaller learning rate."
                    )

                history[epoch] = self._mse(
                    Phi,
                    y,
                    theta
                )

            self.theta = theta
            self.loss_history_ = history

        elif method == "sgd":
            rng = np.random.default_rng(random_state)

            theta = np.zeros(2)
            history = np.empty(epochs)

            for epoch in range(epochs):
                order = rng.permutation(n)

                for i in order:
                    phi_i = Phi[i]
                    residual_i = phi_i @ theta - y[i]

                    grad_i = 2.0 * residual_i * phi_i

                    theta -= lr * grad_i

                    if not np.all(np.isfinite(theta)):
                        raise FloatingPointError(
                            "SGD became numerically unstable. "
                            "Try a smaller learning rate."
                        )

                history[epoch] = self._mse(
                    Phi,
                    y,
                    theta
                )

            self.theta = theta
            self.loss_history_ = history

        else:
            raise ValueError(
                "method must be 'closed_form', 'batch_gd', or 'sgd'"
            )

        return self

    def predict(self, x):
        if self.theta is None:
            raise RuntimeError("Model must be fitted before prediction.")

        Phi = self._design_matrix(x)
        return Phi @ self.theta

    @property
    def intercept_(self):
        if self.theta is None:
            raise RuntimeError("Model must be fitted first.")
        return self.theta[0]

    @property
    def slope_(self):
        if self.theta is None:
            raise RuntimeError("Model must be fitted first.")
        return self.theta[1]


Prepare regression variables

In [4]:
x_feat = np.log10(d_meas / d0)
y_target = PL_meas

Fit the closed-form model

In [5]:
model_cf = MyLinearRegression().fit(
    x_feat,
    y_target,
    method="closed_form"
)


Estimate shadowing standard deviation

In [6]:
def estimate_sigma(model, x, y):
    """Estimate shadowing sigma from model residuals."""
    residuals = y - model.predict(x)

    sigma2_hat = np.mean(residuals ** 2)
    sigma_hat = np.sqrt(sigma2_hat)

    return sigma_hat, residuals


sigma_cf, residuals_cf = estimate_sigma(
    model_cf,
    x_feat,
    y_target
)

In [7]:
Pt_watts = 100.0

# Convert watts -> milliwatts -> dBm
Pt_dBm = 10 * np.log10(Pt_watts * 1000)

d_query = 2000.0  # 2 km, in metres

if d_query <= 0:
    raise ValueError("Distance must be greater than zero.")


# Check whether the requested distance is outside
# the measured data range.
if d_query > d_meas.max():
    print(
        f"\nNote: d = {d_query:.0f} m is beyond the maximum "
        f"measured distance ({d_meas.max():.1f} m)."
    )
    print(
        "The prediction is therefore an extrapolation beyond "
        "the fitted measurement range."
    )


# Feature corresponding to d = 2000 m
x_query = np.log10(d_query / d0)

# Deterministic path-loss prediction
PL_pred_dB = model_cf.predict(
    np.array([x_query])
)[0]

# Mean received power
Pr_mean_dBm = Pt_dBm - PL_pred_dB


print("\n" + "=" * 60)
print("QUESTION 5 — RECEIVED POWER AT 2 km")
print("=" * 60)

print(
    f"Transmit power : {Pt_watts:.1f} W"
)
print(
    f"Transmit power : {Pt_dBm:.3f} dBm"
)
print(
    f"Distance       : {d_query:.0f} m"
)
print(
    f"Path loss      : {PL_pred_dB:.3f} dB"
)
print(
    f"Mean received power : {Pr_mean_dBm:.3f} dBm"
)



Note: d = 2000 m is beyond the maximum measured distance (1764.3 m).
The prediction is therefore an extrapolation beyond the fitted measurement range.

QUESTION 5 — RECEIVED POWER AT 2 km
Transmit power : 100.0 W
Transmit power : 50.000 dBm
Distance       : 2000 m
Path loss      : 141.561 dB
Mean received power : -91.561 dBm


Probability that received power exceeds -95 dBm

In [8]:
threshold_dBm = -95.0
if sigma_cf <= 0 or not np.isfinite(sigma_cf):
    raise ValueError(
        "Estimated shadowing standard deviation is invalid."
    )


z = (
    threshold_dBm - Pr_mean_dBm
) / sigma_cf

# P(P_r > threshold)
prob_above_threshold = stats.norm.sf(z)


print("\n" + "-" * 60)
print("PROBABILITY OF RECEIVED POWER > -95 dBm")
print("-" * 60)

print(
    f"Threshold              : {threshold_dBm:.1f} dBm"
)

print(
    f"Shadowing std. dev.    : {sigma_cf:.3f} dB"
)


------------------------------------------------------------
PROBABILITY OF RECEIVED POWER > -95 dBm
------------------------------------------------------------
Threshold              : -95.0 dBm
Shadowing std. dev.    : 5.800 dB
